# GPT-2 124M — DIMER text-generation tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/gpt2-text-generation-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/tutorials/gpt2_text_generation_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-openai--community%2Fgpt2-ffcc4d?style=flat)](https://huggingface.co/openai-community/gpt2)
[![Upstream](https://img.shields.io/badge/Upstream-openai%2Fgpt--2-181717?style=flat&logo=github&logoColor=white)](https://github.com/openai/gpt-2)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** causal text generation (continuing one English prompt) using the pinned GPT-2 124M weights, with greedy decoding by default and explicit, seeded nucleus sampling on request

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`GPT2TextGenerationPipeline`) rather than reimplementing model inference. At inference the byte-level BPE tokenizer turns the prompt into token ids (no special tokens are added), and the 12-layer decoder-only Transformer predicts one next-token distribution over the 50,257-token vocabulary at a time, feeding each chosen token back until `max_new_tokens` is reached or the end-of-text token is produced. **Two decoding modes are demonstrated and must not be confused (INF8):** greedy decoding (`do_sample=False`, the pipeline default) takes the argmax at every step and is deterministic on a fixed device and dtype — it is the mode for reproducibility checks and tends to repeat itself; nucleus sampling (`do_sample=True` with `temperature`, `top_p` and a mandatory `seed`) draws from the truncated distribution and is the mode usually preferred for actual use, reproducible only for the same seed on the same host. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. GPT-2 is a **base language model**: no chat template, no instruction following, no safety tuning, English web text of 2019 vintage. What the upstream checkpoint supplies is the model and tokenizer; what this repository adds is manifest verification, input validation and ceilings (prompts are rejected, never truncated), a settings validator that refuses unseeded sampling, the fixed pad/EOS handling, and a fixed output contract. **No quality metric exists** for a free-text continuation without a reference corpus; the pipeline ships no metric helper.

**Learning objectives:** bootstrap the repository in a fresh runtime, author a synthetic prompt (or upload your own), surface the pipeline's ceilings and validate both decoding settings before the model runs, stage and digest-verify the immutable upstream snapshot, generate a greedy continuation and confirm it is deterministic, generate a seeded sampled continuation with every setting echoed and confirm the seed reproduces it, read `finished_by` and the pad/EOS quirk correctly, understand why no metric is reported and what corpus a perplexity number would need, and export machine-readable results plus provenance.

**This notebook does not demonstrate:** chat or instruction following (GPT-2 has neither), batching (one prompt per call), raw logits or hidden states, fine-tuning, beam search, non-English text, or any content filtering. The repository exposes none of these.


## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32). The model card's CPU smoke verified the 15-file snapshot in 0.32 s, loaded in 4.25 s, produced 32 greedy tokens from a 7-token prompt in 0.67 s and two seeded 16-token samples in 0.59 s together, so the default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 548 MB `model.safetensors` are the largest downloads of the run.
- **Knowledge:** basic Python; what next-token prediction is; the difference between argmax decoding and sampling from a truncated distribution.
- **Data:** the default sample is one synthetic English prompt authored in code; BYOD is one UTF-8 text file, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API. A base language model can continue any prompt with false, biased or offensive text — read the output before reusing it.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing snapshot file from the Hugging Face Hub at the immutable revision. No credentials are needed.


## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `tokenizers`, `huggingface-hub`, `safetensors`, `numpy`) are pinned exactly by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on both CPU and CUDA; no compilation or quantisation is applied. Look for a dictionary reporting the repository revision, Python, `torch`, `transformers` and `numpy` versions, and whether CUDA is available.


In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/gpt2-text-generation-pipeline.git'
REPO_NAME = 'gpt2-text-generation-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, numpy, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Author the synthetic prompt or optional BYOD

The default sample is **synthetic**: one short English prompt written in this cell — the same prompt the model card's CPU smoke used — so it needs no download and contains no personal data. It ships **no reference continuation**, so whatever the model produces is smoke/sanity evidence that the code path works, never a quality measurement and never benchmark evidence. The decoding settings are Colab form parameters: `GREEDY_MAX_NEW_TOKENS` for the deterministic default, and `SAMPLE_MAX_NEW_TOKENS`, `TEMPERATURE`, `TOP_P`, `SEED` for the sampling demonstration; they are validated against the package in Section 3 and echoed back by the pipeline in Sections 5 and 6.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file whose whole content (leading and trailing whitespace stripped) is the prompt — at most `MAX_TEXT_CHARS` characters and at most `MAX_PROMPT_TOKENS` BPE tokens, with prompt plus new tokens inside `CONTEXT_LENGTH` (over-long prompts are rejected by the pipeline, not truncated). The upload stays inside this runtime.


In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
GREEDY_MAX_NEW_TOKENS = 32  # @param {type:"integer"}
SAMPLE_MAX_NEW_TOKENS = 16  # @param {type:"integer"}
TEMPERATURE = 0.8  # @param {type:"number"}
TOP_P = 0.9  # @param {type:"number"}
SEED = 7  # @param {type:"integer"}
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    prompt = io.TextIOWrapper(io.BytesIO(uploaded[sample_name]), encoding='utf-8').read().strip()
    sample_kind = 'BYOD upload'
else:
    prompt = 'The weather in the mountains is usually'
    sample_name = 'synthetic_weather_prompt'
    sample_kind = 'synthetic (authored in this cell; the model card smoke prompt)'
prompt_sha256 = hashlib.sha256(prompt.encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'chars': len(prompt), 'prompt_sha256': prompt_sha256})
print(repr(prompt[:200]))

## 3. Validate the prompt and both decoding settings against the pipeline ceilings

The pipeline enforces its ceilings inside `generate`; this cell imports the same constants from the package so the values shown are the ones in force. `CONTEXT_LENGTH` (1024) is the model's positional window and bounds prompt tokens plus new tokens; `MAX_PROMPT_TOKENS` (1023) leaves room for at least one generated token; `MAX_NEW_TOKENS` (256) bounds one call's cost; `MAX_TEXT_CHARS` is the character guard applied before tokenisation; `VOCAB_SIZE` is the output vocabulary; `EOS_TOKEN_ID` and `PAD_TOKEN_ID` are equal by construction — **GPT-2 ships no pad token, so the pipeline fixes `pad_token_id = eos_token_id = 50256` in code** and passes an all-ones attention mask, which is why the single-prompt path raises no padding warning and why a generated `50256` means "end of text", after which the completion is cut. The token count of the prompt can only be checked after tokenisation, which needs the loaded tokenizer, so that ceiling is confirmed by the model call in Section 5 (the pipeline rejects, never truncates).

Both decoding configurations are validated here through the package's public `validate_settings` helper — the same function `generate` calls — so a bad `temperature`, `top_p`, `max_new_tokens` or a missing `seed` fails **before** any model work, naming the condition. The returned dicts are the canonical settings the pipeline will echo back: for greedy decoding `temperature`, `top_p` and `seed` are recorded as `None` because they play no role.


In [ ]:
from gpt2_text_generation_pipeline import CONTEXT_LENGTH, DEFAULT_MAX_NEW_TOKENS, EOS_TOKEN_ID, MAX_NEW_TOKENS, MAX_PROMPT_TOKENS, MAX_TEXT_CHARS, PAD_TOKEN_ID, VOCAB_SIZE, validate_settings

ceilings = {'CONTEXT_LENGTH': CONTEXT_LENGTH, 'MAX_PROMPT_TOKENS': MAX_PROMPT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'VOCAB_SIZE': VOCAB_SIZE, 'EOS_TOKEN_ID': EOS_TOKEN_ID, 'PAD_TOKEN_ID': PAD_TOKEN_ID}
print(ceilings)
print({'pad_eos_quirk': f'GPT-2 has no pad token; PAD_TOKEN_ID == EOS_TOKEN_ID == {PAD_TOKEN_ID}: {PAD_TOKEN_ID == EOS_TOKEN_ID}'})
problems = []
if not prompt.strip():
    problems.append('prompt is empty: supply a non-blank prompt')
if len(prompt) > MAX_TEXT_CHARS:
    problems.append(f'prompt has {len(prompt)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}: shorten the prompt')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
greedy_settings = validate_settings(GREEDY_MAX_NEW_TOKENS, False, 1.0, 1.0, None)
sampling_settings = validate_settings(SAMPLE_MAX_NEW_TOKENS, True, TEMPERATURE, TOP_P, SEED)
print({'greedy_settings': greedy_settings})
print({'sampling_settings': sampling_settings})
print({'chars': len(prompt), 'within_ceilings': True, 'token_ceiling': f'prompt tokens <= MAX_PROMPT_TOKENS={MAX_PROMPT_TOKENS} and prompt + new tokens <= CONTEXT_LENGTH={CONTEXT_LENGTH} are checked by the pipeline after tokenisation; it rejects, never truncates'})

## 4. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here) and refuses remote model code (`trust_remote_code=False`). The Git repository commits the DIMER snapshot manifest (`weights/gpt2/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each of the 15 snapshot files) and the small config, tokenizer and upstream ONNX-config files, but git-ignores the 548 MB `model.safetensors`, so a fresh clone must stage the missing file first. `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every listed file and raises on the first size or digest mismatch; its returned dict is summarised. Only afterwards does `from_pretrained(weights_dir=WEIGHTS_DIR)` load tokenizer and model from that verified directory with `local_files_only=True` in float32 — there is no fallback to a different download. The effective model identity and the selected device are printed before inference.


In [ ]:
from gpt2_text_generation_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, GPT2TextGenerationPipeline, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot = verify_snapshot(WEIGHTS_DIR)
print({'snapshot_path': snapshot['path'], 'model_id': snapshot['modelId'], 'revision': snapshot['revision'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')})
pipe = GPT2TextGenerationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'dtype': 'float32', 'source': pipe.source})

## 5. Generate with the greedy default and confirm it is deterministic

`generate(prompt, max_new_tokens=...)` with the default `do_sample=False` takes the argmax at every step. It returns `completion` (the new text only), `text` (prompt plus completion), `prompt_tokens`, `new_tokens`, `finished_by` (`'eos'` when the model produced the end-of-text token 50256 — the completion is cut there — or `'max_new_tokens'` when the budget ran out), the echoed `settings` (`decoding: 'greedy'`, with `temperature`/`top_p`/`seed` as `None`), the device and the model identity. **Greedy decoding is deterministic on a fixed device and dtype:** the cell calls `generate` twice with the same settings and checks the two completions are byte-identical — a falsifiable check of the reproducibility contract, which does not extend across devices, PyTorch builds or dtypes. Greedy output is also the mode that repeats itself and drifts into generic text; it is the reference mode, not the recommended one for actual use.

**Evaluation:** the repository ships **no metric helper and reports no performance measure**. A continuation has no ground truth; the standard intrinsic measure, perplexity, needs a held-out reference corpus the caller supplies, and any quality judgement needs human raters or a downstream task with labels. The synthetic prompt has no reference, so **no metric is reported**; the model card's smoke observation on this prompt (32 greedy tokens, `finished_by = 'max_new_tokens'`, completion beginning `" good, but the snow is not."`) is one measurement on that host, not an expected value — near-tied logits can flip a token between CPU and CUDA kernels and change everything after it.


In [ ]:
import time

started = time.perf_counter()
greedy = pipe.generate(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS)
greedy_elapsed = time.perf_counter() - started
greedy_repeat = pipe.generate(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS)
greedy_checks = {
    'settings_echoed_as_validated': greedy['settings'] == greedy_settings,
    'decoding_is_greedy': greedy['settings']['decoding'] == 'greedy',
    'new_tokens_within_budget': 0 <= greedy['new_tokens'] <= GREEDY_MAX_NEW_TOKENS,
    'prompt_tokens_within_ceiling': 1 <= greedy['prompt_tokens'] <= MAX_PROMPT_TOKENS,
    'text_is_prompt_plus_completion': greedy['text'] == greedy['prompt'] + greedy['completion'],
    'finished_by_is_known': greedy['finished_by'] in ('eos', 'max_new_tokens'),
    'greedy_repeat_is_identical': greedy_repeat['completion'] == greedy['completion'],
}
if not all(greedy_checks.values()):
    raise RuntimeError(f'greedy generate output failed a sanity check: {greedy_checks}')
print({key: value for key, value in greedy.items() if key not in ('prompt', 'completion', 'text')})
print({'seconds_first_call': round(greedy_elapsed, 3), 'checks': greedy_checks})
print(f'prompt:     {prompt!r}')
print(f"completion: {greedy['completion']!r}")
metrics = {}
print('no metric is reported: a continuation has no ground truth, the sample has no reference corpus, and the repository ships no metric helper; perplexity needs a held-out corpus you supply')

## 6. Generate with explicit, seeded nucleus sampling

Sampling is opt-in: `do_sample=True` with `temperature` (rescales the logits; below 1 sharpens, above 1 flattens), `top_p` (nucleus truncation: only the smallest set of tokens whose cumulative probability reaches `top_p` is sampled from) and a **mandatory `seed`** — `validate_settings` refuses unseeded sampling so a sampled result is always reproducible for the same seed on the same host. The pipeline seeds PyTorch's generator immediately before the model call. Every setting is echoed in `settings` (`decoding: 'nucleus-sampling'`) so an exported result records exactly how it was produced (INF9). This cell samples twice with the same seed and checks the completions are identical, then contrasts the sampled completion with the greedy one from Section 5: a different completion is expected (the model card observed `" good, but you need to be careful with your gear and don't go to"` for seed 7 on its host — an observation, not an expected value, because the sampled path depends on the exact floating-point logits of that host). Sampling is the mode usually preferred for actual use because it avoids greedy repetition; it is not "better" in any measured sense here, and a different seed gives a different continuation.


In [ ]:
started = time.perf_counter()
sampled = pipe.generate(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED)
sampled_elapsed = time.perf_counter() - started
sampled_repeat = pipe.generate(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED)
sampled_checks = {
    'settings_echoed_as_validated': sampled['settings'] == sampling_settings,
    'decoding_is_nucleus_sampling': sampled['settings']['decoding'] == 'nucleus-sampling',
    'seed_echoed': sampled['settings']['seed'] == SEED,
    'new_tokens_within_budget': 0 <= sampled['new_tokens'] <= SAMPLE_MAX_NEW_TOKENS,
    'text_is_prompt_plus_completion': sampled['text'] == sampled['prompt'] + sampled['completion'],
    'same_seed_reproduces': sampled_repeat['completion'] == sampled['completion'],
}
if not all(sampled_checks.values()):
    raise RuntimeError(f'sampled generate output failed a sanity check: {sampled_checks}')
print({key: value for key, value in sampled.items() if key not in ('prompt', 'completion', 'text')})
print({'seconds_first_call': round(sampled_elapsed, 3), 'checks': sampled_checks})
print(f"greedy:  {greedy['completion'][:120]!r}")
print(f"sampled: {sampled['completion'][:120]!r}")
print({'sampled_differs_from_greedy': sampled['completion'] != greedy['completion'][: len(sampled['completion'])], 'note': 'expected but not asserted; both are unscored continuations'})

## 7. Export outputs and provenance

One JSON record is written under `outputs/`: the prompt, the greedy result and the sampled result (each with completion, token counts, `finished_by`, and the echoed settings, so every completion carries the exact decoding configuration that produced it), the determinism and sanity checks, the ceilings in force including the pad/EOS ids, the empty metric block, the sample identity and digest, the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, `numpy`, device, dtype). No credentials are involved in any step, so none can reach the export.


In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
payload = {
    'prompt': prompt,
    'greedy': {key: greedy[key] for key in ('completion', 'text', 'prompt_tokens', 'new_tokens', 'finished_by', 'settings')},
    'sampled': {key: sampled[key] for key in ('completion', 'text', 'prompt_tokens', 'new_tokens', 'finished_by', 'settings')},
    'sanity_checks': {'greedy': greedy_checks, 'sampled': sampled_checks},
    'seconds': {'greedy_first_call': round(greedy_elapsed, 3), 'sampled_first_call': round(sampled_elapsed, 3)},
    'ceilings': ceilings,
    'metrics': metrics,
    'sample': {'name': sample_name, 'kind': sample_kind, 'prompt_sha256': prompt_sha256},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'numpy': numpy.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/gpt2_text_generation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/gpt2_text_generation_result.json')

## Interpretation and limits

Both completions are unscored continuations from a 2019 base language model: they can be false, repetitive, biased or offensive, they carry no confidence or probability, and nothing in the pipeline filters them. Greedy decoding is the deterministic reference mode (identical on repeat, on the same device and dtype); seeded nucleus sampling is the mode usually preferred for use (identical on repeat for the same seed on the same host, different for a different seed or host). Neither is "better" in any measured sense here: no metric is reported because none can be computed without a reference corpus (perplexity) or human judgements, and the model card's smoke completions are observations from one host, not expected values. Prompts are rejected above `MAX_PROMPT_TOKENS` or when prompt plus new tokens would exceed the 1024-token window, never truncated; `finished_by = 'eos'` means the model emitted token 50256, which doubles as the pad id because GPT-2 ships no pad token; the pipeline exposes one prompt per call, no chat format, no batching, no logits and no fine-tuning.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated prompt and both decoding settings against the enforced ceilings, execute the public pipeline path in both decoding modes with the stated determinism properties, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, text quality on any domain, factual reliability, safety for high-consequence use, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/gpt2/` and rerun Section 4. A `ValueError`/`TypeError` from `validate_settings` in Section 3: a form parameter is out of range (`max_new_tokens` 1..256, `temperature` > 0, `top_p` in (0, 1], integer `seed` >= 0) — fix it and rerun from Section 2. A `ValueError` naming `MAX_PROMPT_TOKENS` or `CONTEXT_LENGTH` in Section 5: the BYOD prompt tokenises too long for the requested budget — shorten it or lower `GREEDY_MAX_NEW_TOKENS`. A `greedy_repeat_is_identical` failure would indicate non-deterministic kernels on the host and should be reported with the runtime identity.

**Next experiments.** Change `SEED` and rerun Section 6 to see a different sampled continuation; set `TEMPERATURE` to 0.3 and 1.5 and compare how conservative or erratic the samples become; raise `GREEDY_MAX_NEW_TOKENS` to 128 and watch greedy decoding repeat itself; upload a paragraph via `USE_BYOD` and inspect `prompt_tokens`; compute perplexity on a small held-out text of your own with your own code as the first step towards an intrinsic number. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/openai-community/gpt2
- Upstream code: https://github.com/openai/gpt-2
- Language Models are Unsupervised Multitask Learners (Radford et al., 2019; OpenAI technical report, no arXiv identifier): https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
